# Một sản phẩm, nhiều tên: chuẩn hoá và dedupe fuzzy trước khi xếp hạng sản phẩm

Bộ `online-retail-ii`, sheet **`Year 2010-2011`** (541 910 dòng). Head of Merchandising muốn bảng **top sản phẩm theo
doanh thu** cho buổi họp chọn hàng mùa sau, và hỏi thêm: "Hình như cùng một món mà hệ thống ghi nhiều tên, các bạn gộp
lại giúp nhé."

Có hai câu hỏi dedupe khác nhau, và nhầm hai câu này là lỗi hay gặp nhất:

1. **Cùng mã hàng, nhiều mô tả** (`StockCode` giống nhau): mã là khoá, mô tả chỉ là nhãn. Chọn một tên chuẩn cho mỗi
   mã là đủ.
2. **Khác mã hàng, tên gần giống nhau**: có phải cùng sản phẩm bị tạo mã hai lần? Fuzzy matching cho ra ứng viên;
   bạn phải kiểm ứng viên có thật là trùng không trước khi gộp tiền của hai mã.

Việc của bạn: chuẩn hoá mô tả, chọn tên chuẩn theo mã, tìm cặp tên gần giống bằng fuzzy matching có blocking, lọc
ứng viên bằng quy tắc, rồi **làm phần trắc nghiệm trong app**.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm hoặc hằng. Điền vào chỗ `# TODO`; hàm không được sửa DataFrame đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/06-data-cleaning/exercises/intermediate/01-product-name-dedupe -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc sheet 2010-2011
# xlsx đọc mất 30-60 giây: lần đầu lưu bản parquet vào thư mục tạm, các lần sau đọc trong vài giây
import os
import sys
import tempfile
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd


def read_sheet(sheet):
    src = ensure("online-retail-ii") / "online_retail_II.xlsx"
    st = src.stat()
    cache = Path(tempfile.gettempdir()) / "da-cache" / f"online_retail_II-{sheet.replace(' ', '_')}-{st.st_size}-{st.st_mtime_ns}.parquet"
    if not cache.exists():
        df = pd.read_excel(src, sheet_name=sheet, dtype={"Invoice": str, "StockCode": str, "Description": str})
        cache.parent.mkdir(parents=True, exist_ok=True)
        tmp = cache.with_name(f"{cache.name}.{os.getpid()}.tmp")
        df.to_parquet(tmp, index=False)
        os.replace(tmp, cache)  # ghi atomic: không để lại tệp dở dang
    return pd.read_parquet(cache)


tx = read_sheet("Year 2010-2011")
print(tx.shape)

In [ ]:
import difflib
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

FEE_CODES = {
    "POST", "DOT", "M", "C2", "D", "S", "B", "BANK CHARGES", "AMAZONFEE", "CRUK", "PADS",
    "ADJUST", "ADJUST2", "TEST001", "TEST002",
}


def valid_lines(tx: pd.DataFrame) -> pd.DataFrame:
    """Cho sẵn (không phải TODO): dòng hợp lệ theo định nghĩa dùng chung, StockCode viết hoa, bỏ mã phí, cột revenue."""
    out = tx.copy()
    out["Invoice"] = out["Invoice"].astype(str)
    out["StockCode"] = out["StockCode"].astype(str).str.strip().str.upper()
    keep = ~out["Invoice"].str.startswith("C") & (out["Quantity"] > 0) & (out["Price"] > 0) & ~out["StockCode"].isin(FEE_CODES)
    out = out[keep]
    return out.assign(revenue=out["Quantity"] * out["Price"])

In [ ]:
# explore: viết hoa StockCode gộp bao nhiêu mã? (vd. 85123a và 85123A là một sản phẩm)
valid = valid_lines(tx)
raw_code = tx.loc[valid.index, "StockCode"].astype(str).str.strip()
print(f"{len(valid):,} dòng; mã hàng giữ hoa thường: {raw_code.nunique():,}; sau khi viết hoa: {valid.StockCode.nunique():,}")
print(f"mô tả gốc khác nhau: {valid.Description.nunique():,}")

## 1. Chuẩn hoá mô tả

`normalize_description(s) -> Series`: viết hoa, thay mọi chuỗi ký tự **không phải** chữ `A-Z`, số `0-9` hoặc `&`
bằng **một** khoảng trắng, bỏ khoảng trắng hai đầu. Ô thiếu giữ là thiếu, index giữ nguyên.

Ví dụ: `"PAPER CRAFT , LITTLE BIRDIE"` → `"PAPER CRAFT LITTLE BIRDIE"`; `"SET/6 RED SPOTTY CUPS"` →
`"SET 6 RED SPOTTY CUPS"`; `"50'S CHRISTMAS  KIT"` → `"50 S CHRISTMAS KIT"`.

In [ ]:
def normalize_description(s: pd.Series) -> pd.Series:
    """Viết hoa, ký tự ngoài [A-Z0-9&] thành một khoảng trắng, strip; thiếu giữ là thiếu."""
    return s.astype("string").str.upper().str.replace(r"[^A-Z0-9&]+", " ", regex=True).str.strip()

## 2. Tên chuẩn cho mỗi mã hàng

`canonical_names(lines) -> DataFrame`, index `StockCode`, cột `name` và `n_variants`:

- chuẩn hoá `Description`, bỏ dòng mô tả thiếu;
- `name`: mô tả (đã chuẩn hoá) xuất hiện trên **nhiều dòng nhất** của mã đó; hoà thì lấy tên nhỏ nhất theo thứ tự
  chữ cái (để kết quả tái lập được);
- `n_variants`: số mô tả khác nhau (sau chuẩn hoá) của mã.

In [ ]:
def canonical_names(lines: pd.DataFrame) -> pd.DataFrame:
    """Tên chuẩn (mô tả nhiều dòng nhất, hoà lấy nhỏ nhất) và số biến thể theo StockCode."""
    d = pd.DataFrame({"StockCode": lines["StockCode"].to_numpy(),
                      "desc": normalize_description(lines["Description"]).to_numpy()}).dropna()
    counts = (d.groupby(["StockCode", "desc"]).size().rename("n").reset_index()
              .sort_values(["StockCode", "n", "desc"], ascending=[True, False, True]))
    first = counts.drop_duplicates("StockCode").set_index("StockCode")
    return pd.DataFrame({"name": first["desc"], "n_variants": counts.groupby("StockCode").size()})

In [ ]:
# explore: mã nào có nhiều tên nhất, và gom theo mô tả thay vì theo mã làm lệch doanh thu bao nhiêu?
names = canonical_names(valid)
print(f"{(names.n_variants > 1).sum()} mã có hơn một mô tả; phân bố:", names.n_variants.value_counts().sort_index().to_dict())
v = valid.assign(desc=normalize_description(valid.Description)).join(names["name"], on="StockCode")
off = v.desc != v.name
print(f"doanh thu trên dòng có mô tả khác tên chuẩn: {v.revenue[off].sum():,.0f} GBP ({v.revenue[off].sum() / v.revenue.sum():.1%})")
for code in ["23196", "23203", "22502"]:
    print(code, v[v.StockCode == code].groupby("desc").agg(lines=("revenue", "size"), revenue=("revenue", "sum"),
                                                           median_price=("Price", "median")).round(2).to_dict("index"))

## 3. Fuzzy matching có blocking

So mọi cặp trong khoảng 3 800 tên là hơn 7 triệu phép so; với `difflib` mất vài phút. **Blocking**: chỉ so các tên
cùng **từ đầu tiên**, giảm số cặp xuống vài chục lần. Đánh đổi: lỗi gõ ở từ đầu tiên (`RETO` / `RETRO`) bị bỏ sót.

`fuzzy_pairs(names, threshold=0.9) -> DataFrame` cột `a`, `b`, `score`:

- lấy các tên **khác nhau**, bỏ thiếu, sắp tăng dần;
- trong mỗi block (cùng từ đầu tiên), so mọi cặp `a < b` bằng
  `difflib.SequenceMatcher(None, a, b, autojunk=False).ratio()`;
- giữ cặp có `score >= threshold`, sắp theo `score` giảm dần rồi `a`, `b` tăng dần, index `0..n-1`;
- không có cặp nào thì trả DataFrame rỗng vẫn đủ ba cột.

In [ ]:
def fuzzy_pairs(names: pd.Series, threshold: float = 0.9) -> pd.DataFrame:
    """Cặp tên cùng từ đầu tiên có SequenceMatcher.ratio() >= threshold."""
    unique = pd.Series(sorted(set(names.dropna())), dtype=object)
    rows = []
    for _, block in unique.groupby(unique.str.split(" ").str[0]):
        items = block.tolist()
        for i, a in enumerate(items):
            for b in items[i + 1:]:
                score = difflib.SequenceMatcher(None, a, b, autojunk=False).ratio()
                if score >= threshold:
                    rows.append((a, b, score))
    out = pd.DataFrame(rows, columns=["a", "b", "score"])
    return out.sort_values(["score", "a", "b"], ascending=[False, True, True], ignore_index=True)

In [ ]:
# explore: 30 cặp giống nhau nhất giữa các mã KHÁC nhau. Có cặp nào thật sự là một sản phẩm?
pairs = fuzzy_pairs(names["name"], threshold=0.9)
print(len(pairs), "cặp")
pairs.head(30)

## 4. Lọc ứng viên bằng quy tắc

Đọc 30 cặp trên: `1 WICK` / `3 WICK`, `30CM` / `60CM`, `LETTER A` / `LETTER B`, `JUG` / `MUG` giống nhau tới 0.96
nhưng là **sản phẩm khác nhau** (cỡ, màu, chữ cái). Điểm similarity cao không có nghĩa là trùng.

`likely_same(a, b) -> bool`: tách hai tên thành túi từ (đếm cả từ lặp), lấy các từ chỉ có ở `a` và chỉ có ở `b`.

- Không có từ khác nhau (chỉ đổi thứ tự): `True`.
- Đúng **một** từ khác ở mỗi bên, hai từ đó **không chứa chữ số**, đều dài **ít nhất 4** ký tự và
  `SequenceMatcher(None, x, y).ratio() >= 0.8` (lỗi chính tả như `RETRO` / `RETO`, `COLOUR` / `COLOR`): `True`.
- Mọi trường hợp khác: `False`.

In [ ]:
def likely_same(a: str, b: str) -> bool:
    """Hai tên có khả năng là cùng một sản phẩm (đổi thứ tự từ hoặc lỗi chính tả một từ)."""
    ca, cb = Counter(a.split()), Counter(b.split())
    only_a, only_b = list((ca - cb).elements()), list((cb - ca).elements())
    if not only_a and not only_b:
        return True
    if len(only_a) != 1 or len(only_b) != 1:
        return False
    x, y = only_a[0], only_b[0]
    if any(ch.isdigit() for ch in x + y):
        return False
    return min(len(x), len(y)) >= 4 and difflib.SequenceMatcher(None, x, y).ratio() >= 0.8

In [ ]:
# explore: còn bao nhiêu ứng viên? Hạ ngưỡng xuống 0.85 thì sao?
for t in (0.9, 0.85):
    p = fuzzy_pairs(names["name"], threshold=t)
    keep = p[[likely_same(a, b) for a, b in zip(p.a, p.b)]]
    print(f"ngưỡng {t}: {len(p)} cặp, {len(keep)} qua quy tắc")
    if len(keep):
        print(keep.assign(code_a=keep.a.map(names.reset_index().groupby("name").StockCode.first()),
                          code_b=keep.b.map(names.reset_index().groupby("name").StockCode.first())).to_string())

In [ ]:
# explore: top 10 sản phẩm theo doanh thu, gom theo mã + tên chuẩn
top = (v.groupby("StockCode").agg(name=("name", "first"), revenue=("revenue", "sum"), variants=("desc", "nunique"))
       .sort_values("revenue", ascending=False).head(10))
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(top["name"], top["revenue"] / 1000, color="#4c78a8")
ax.invert_yaxis()
ax.set_xlabel("nghìn GBP")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Top 10 sản phẩm 12/2010-12/2011, gom theo mã hàng (tên chuẩn)", loc="left")
plt.show()
top

## 5. Kết luận cho Head of Merchandising, rồi làm trắc nghiệm

Tự trả lời trước khi làm trắc nghiệm:

1. Xếp hạng nên gom theo khoá nào? Gom theo mô tả gốc làm mã nào bị tách thành nhiều dòng?
2. Fuzzy matching giữa các mã khác nhau tìm ra bao nhiêu trùng thật? Bạn có gộp tiền của hai mã nào không?
3. Mã `22502` có hai mô tả `PICNIC BASKET WICKER SMALL` và `PICNIC BASKET WICKER 60 PIECES` với giá trung vị rất
   khác nhau. Cùng mã có chắc là cùng một món hàng không? Bạn sẽ hỏi ai?

Xong thì mở app, làm phần **trắc nghiệm** của bài.